# Quadratic $F_A$ reweighting and axial-form-factor priors

This notebook validates the production from the effective-$M_A$ interpolation to a quadratic fit in $F_A$, checks the coefficient-count/CV fix, and pulls the new fits from [arXiv:2512.14097](https://arxiv.org/abs/2512.14097).

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ROOT

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/zexp_reweighting.py').exists())
sys.path.insert(0, str(repo_root))

from src.zexp_reweighting import (
    AXIAL_FORM_FACTOR_Q2_ZERO, MA_CCQE_GRID_GEV, ZEXP_SIGMA_VALUES,
    MINERVA_LEGACY_PRIOR, ZEXP_PRIORS, MINERVA_K6_PRIOR,
    MINERVA_K6_DIAGONAL_PRIOR, axial_form_factor_zexp,
    complete_zexp_a_values,
    effective_axial_mass_gev, interpolate_ma_spline_weights,
    quadratic_fa_spline_weights, compute_zexp_prior_weights,
    compute_zexp_weights,
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold'})
validation_file = Path('/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root')
assert validation_file.exists()
new_correlated_priors = ZEXP_PRIORS[2:6]

## A representative real-event sample

The sample is restricted to rows with physical true $Q^2$ and a finite, responsive seven-point `MaCCQE_UBGenie` spline. Non-responsive rows have no axial-mass information and are uninformative for this validation.

In [ ]:
sample_raw = ROOT.RDataFrame('tree', str(validation_file)).Range(100_000).AsNumpy(
    ['GTruth_gQ2', 'MaCCQE_UBGenie']
)
sample_q2_all = np.asarray(sample_raw['GTruth_gQ2'], dtype=float)
sample_ma_all = np.stack([np.asarray(row, dtype=float) for row in sample_raw['MaCCQE_UBGenie']])
responsive = (
    np.isfinite(sample_q2_all) & (sample_q2_all >= 0)
    & np.isfinite(sample_ma_all).all(axis=1) & (sample_ma_all[:, 3] > 0)
    & (np.ptp(sample_ma_all, axis=1) > 1e-5)
)
sample_q2 = sample_q2_all[responsive]
sample_ma = sample_ma_all[responsive]
print(f'{len(sample_q2):,} responsive events retained from {len(sample_q2_all):,} rows')
print(f'true Q2 range: {sample_q2.min():.5g} to {sample_q2.max():.5g} GeV^2')

## Why a quadratic in $F_A$ is appropriate

At fixed event kinematics the QE cross section is quadratic in $F_A$. The plot below refits every real event's seven spline samples and evaluates the fit back at those samples. The small residuals verify that the available GENIE splines follow the expected quadratic form.

In [ ]:
fa_grid = AXIAL_FORM_FACTOR_Q2_ZERO / (
    1 + sample_q2[:, None] / MA_CCQE_GRID_GEV[None, :]**2
)**2
quadratic_at_grid = np.column_stack([
    quadratic_fa_spline_weights(sample_q2, fa_grid[:, i], sample_ma)
    for i in range(len(MA_CCQE_GRID_GEV))
])
max_abs_residual = np.max(np.abs(quadratic_at_grid - sample_ma), axis=1)
max_relative_residual = max_abs_residual / np.maximum(np.max(np.abs(sample_ma), axis=1), 1e-12)
print('Maximum relative residual percentiles:')
for p, value in zip([50, 90, 95, 99, 99.9, 100], np.percentile(max_relative_residual, [50, 90, 95, 99, 99.9, 100])):
    print(f'  {p:5g}%: {100*value:.5f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
positive_residuals = max_relative_residual[max_relative_residual > 0]
axes[0].hist(positive_residuals, bins=np.geomspace(1e-8, max(positive_residuals.max(), 1e-7), 70), color='C0')
axes[0].set(xscale='log', yscale='log', xlabel='Maximum relative residual per event', ylabel='Events', title='Quadratic reproduces the seven spline points')
example_i = int(np.argsort(max_relative_residual)[len(max_relative_residual)//2])
order = np.argsort(fa_grid[example_i])
dense_fa = np.linspace(fa_grid[example_i].min(), fa_grid[example_i].max(), 300)
dense_weights = quadratic_fa_spline_weights(
    np.full(len(dense_fa), sample_q2[example_i]), dense_fa,
    np.repeat(sample_ma[example_i][None, :], len(dense_fa), axis=0),
)
axes[1].plot(dense_fa, dense_weights, color='C1', lw=2, label='quadratic fit')
axes[1].plot(fa_grid[example_i, order], sample_ma[example_i, order], 'ko', label='GENIE spline samples')
axes[1].set(xlabel=r'$F_A(Q^2_{true})$', ylabel='Event weight', title=f'Representative event, $Q^2={sample_q2[example_i]:.3f}$ GeV$^2$')
axes[1].legend()
fig.tight_layout()
plt.show()

## Shape preservation in the ordinary interpolation region

Here both methods (inverting the dipole-approximation $F_A$ and the quadratic fit) use exactly the same current MINERvA coefficients. We compare only targets with a finite $M_A^{eff}$ inside the supplied 0.8–1.4 GeV grid, where the old method was expected to behave normally.

In [ ]:
legacy = MINERVA_LEGACY_PRIOR
eigenvalues, eigenvectors = np.linalg.eigh((legacy.covariance + legacy.covariance.T) / 2)
order = np.argsort(eigenvalues)[::-1]
pca_shifts = eigenvectors[:, order] * np.sqrt(np.maximum(eigenvalues[order], 0.0))
pca_comparisons = []

for component_i, pca_shift in enumerate(pca_shifts.T, start=1):
    old_weights = np.empty((len(sample_q2), len(ZEXP_SIGMA_VALUES)))
    quadratic_weights = np.empty_like(old_weights)
    ma_eff = np.empty_like(old_weights)
    for sigma_i, sigma in enumerate(ZEXP_SIGMA_VALUES):
        full = legacy.full_a_values if sigma == 0 else complete_zexp_a_values(
            legacy.free_a_values + sigma*pca_shift, legacy.kmax, legacy.t0_gev2,
            t_cut_gev2=legacy.t_cut_gev2, fa_q2_zero=legacy.fa_q2_zero,
        )
        target_fa = axial_form_factor_zexp(
            sample_q2, full, legacy.t0_gev2, legacy.t_cut_gev2,
        )
        ma_eff[:, sigma_i] = effective_axial_mass_gev(sample_q2, target_fa)
        old_weights[:, sigma_i] = interpolate_ma_spline_weights(
            ma_eff[:, sigma_i], sample_ma,
        )
        quadratic_weights[:, sigma_i] = quadratic_fa_spline_weights(
            sample_q2, target_fa, sample_ma,
        )

    inside = (
        np.isfinite(ma_eff)
        & (ma_eff >= MA_CCQE_GRID_GEV[0])
        & (ma_eff <= MA_CCQE_GRID_GEV[-1])
    )
    common_inside = inside.all(axis=1)
    relative_change = np.abs(
        quadratic_weights[inside] / old_weights[inside] - 1,
    )
    pca_comparisons.append(
        (pca_shift, old_weights, quadratic_weights, ma_eff,
         common_inside, relative_change)
    )
    percentiles = np.percentile(relative_change, [50, 90, 95, 99])
    print(f'PCA{component_i}: {common_inside.sum():,} events remain inside the MA grid for all seven shifts; '
          f'{(~common_inside).sum():,} leave the ordinary region')
    print('  absolute relative change percentiles: ' + ', '.join(
        f'{p}% = {100*value:.4f}%' for p, value in zip([50, 90, 95, 99], percentiles)
    ))

# Keep the PCA1 names used by the pathology study below.
pca1_shift, old_pca1, quadratic_pca1, ma_eff_pca1, common_inside, relative_change = pca_comparisons[0]

fig, axes = plt.subplots(len(pca_comparisons), 4, figsize=(20, 15), squeeze=False)
for component_i, (_, old_weights, quadratic_weights, _, common, changes) in enumerate(pca_comparisons, start=1):
    comparison_subsets = (
        (common, r'$M_A$ valid region'),
        (np.ones(len(sample_q2), dtype=bool), 'all events'),
        (~common, r'$M_A$ invalid region'),
    )
    for column_i, (subset, subset_label) in enumerate(comparison_subsets):
        weight_ax = axes[component_i - 1, column_i]
        weight_ax.plot(
            ZEXP_SIGMA_VALUES, old_weights[subset].mean(axis=0), 'ko-', lw=2,
            label=r'linear in $M_A^{eff}$',
        )
        weight_ax.plot(
            ZEXP_SIGMA_VALUES, quadratic_weights[subset].mean(axis=0),
            'o--', color='C1', lw=2, label=r'quadratic in $F_A$',
        )
        weight_ax.axvline(0, color='0.65', ls=':', lw=1)
        weight_ax.set(
            xlabel=rf'PCA{component_i} shift ($\sigma$)',
            ylabel='Mean event weight',
            title=rf'PCA{component_i}: {subset_label} ($N={subset.sum():,}$)',
        )
        weight_ax.legend(fontsize=8)
    difference_ax = axes[component_i - 1, 3]
    difference_ax.hist(
        changes, bins=np.geomspace(1e-10, max(changes.max(), 1e-9), 70),
        color=f'C{component_i + 1}',
    )
    difference_ax.set(
        xscale='log', yscale='log', xlabel='Absolute relative change',
        ylabel='Evaluations', title=rf'PCA{component_i}: $M_A$ valid event-level difference',
    )
fig.tight_layout()
plt.show()

## New 2025 priors and coefficient validation

In [ ]:
coefficient_rows = []
for prior in new_correlated_priors:
    rebuilt = complete_zexp_a_values(
        prior.free_a_values, prior.kmax, prior.t0_gev2,
        t_cut_gev2=prior.t_cut_gev2, fa_q2_zero=prior.fa_q2_zero,
    )
    coefficient_rows.append({
        'fit': prior.name, 'kmax': prior.kmax,
        'free coefficients': prior.kmax - 4,
        'variation branches': len(prior.variation_branches),
        'max |rebuilt - published CV|': np.max(np.abs(rebuilt - prior.full_a_values)),
        'full production coefficients': np.array2string(prior.full_a_values, precision=8),
    })
pd.DataFrame(coefficient_rows)

In [ ]:
rng = np.random.default_rng(251214097)
q2_curve = np.linspace(0, 2, 300)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, prior in zip(axes.flat, new_correlated_priors):
    central = axial_form_factor_zexp(q2_curve, prior.full_a_values, prior.t0_gev2, prior.t_cut_gev2)
    free_universes = rng.multivariate_normal(prior.free_a_values, prior.covariance, 2000)
    curves = np.array([
        axial_form_factor_zexp(
            q2_curve,
            complete_zexp_a_values(a, prior.kmax, prior.t0_gev2, t_cut_gev2=prior.t_cut_gev2, fa_q2_zero=prior.fa_q2_zero),
            prior.t0_gev2, prior.t_cut_gev2,
        ) for a in free_universes
    ])
    lo, hi = np.percentile(curves, [16, 84], axis=0)
    ax.fill_between(q2_curve, lo, hi, color='C0', alpha=.25, label=r'16–84%')
    ax.plot(q2_curve, central, color='C0', lw=2, label='published CV')
    ax.set(title=prior.name, xlabel=r'$Q^2$ (GeV$^2$)', ylabel=r'$F_A(Q^2)$')
    ax.legend(fontsize=8)
fig.suptitle('New priors in the production negative-$F_A$ convention')
fig.tight_layout()
plt.show()

## Real-event weight diagnostics for all four priors

For each prior we require finite CV and shifted weights, exact agreement between every zero-shift column and the CV, no negative-weight fallback on this responsive sample, and smooth ensemble behavior across all reported PCA directions.

In [ ]:
all_real_weights = compute_zexp_weights(sample_q2, sample_ma)
diagnostic_rows = []
for prior in new_correlated_priors:
    cv = all_real_weights[prior.cv_branch].astype(float)
    variations = [all_real_weights[branch].astype(float) for branch in prior.variation_branches]
    shifted = np.concatenate([values.ravel() for values in variations])
    diagnostic_rows.append({
        'fit': prior.name,
        'all finite': np.isfinite(cv).all() and np.isfinite(shifted).all(),
        'all zero shifts equal CV': all(np.array_equal(values[:, 3], cv) for values in variations),
        'negative weights': int(np.sum(shifted < 0)),
        'shifted p0.1': np.percentile(shifted, .1),
        'shifted median': np.median(shifted),
        'shifted p99.9': np.percentile(shifted, 99.9),
        'shifted maximum': shifted.max(),
    })
diagnostics = pd.DataFrame(diagnostic_rows)
display(diagnostics)
assert diagnostics['all finite'].all()
assert diagnostics['all zero shifts equal CV'].all()
assert (diagnostics['negative weights'] == 0).all()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for ax, prior in zip(axes.flat, new_correlated_priors):
    for component_i, branch in enumerate(prior.variation_branches, start=1):
        values = all_real_weights[branch].astype(float)
        ax.plot(ZEXP_SIGMA_VALUES, values.mean(axis=0), 'o-', lw=1.8, label=f'PCA{component_i}')
    ax.set(title=prior.name, xlabel=r'Parameter shift ($\sigma$)', ylabel='Mean real-event weight')
    ax.legend(fontsize=8)
fig.suptitle('All uncertainty directions on responsive real events')
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
bins = np.linspace(0.5, 3.0, 90)
for ax, prior in zip(axes.flat, new_correlated_priors):
    cv = all_real_weights[prior.cv_branch].astype(float)
    ax.hist(cv, bins=bins, histtype='stepfilled', alpha=.55)
    ax.axvline(1, color='black', ls='--', lw=1)
    ax.set(title=prior.name, xlabel='CV event weight', ylabel='Events', yscale='log')
fig.suptitle('CV weight distributions; the display range is zoomed to the bulk')
fig.tight_layout()
plt.show()

## Diagonal $k_{max}=6$ extraction prior

This companion starts at the MINERvA-hydrogen $k_{max}=6$ CV and retains the reported marginal uncertainties, but sets the $a_1$–$a_2$ covariance to zero. Its two branches shift $a_1$ and $a_2$ directly rather than rotating into PCA directions. The correlated PCA prior above remains unchanged.

In [ ]:
correlated_k6 = MINERVA_K6_PRIOR
diagonal_k6 = MINERVA_K6_DIAGONAL_PRIOR
correlated_cv = all_real_weights[correlated_k6.cv_branch]
diagonal_cv = all_real_weights[diagonal_k6.cv_branch]
assert np.array_equal(correlated_cv, diagonal_cv)

diagonal_variations = [all_real_weights[b].astype(float) for b in diagonal_k6.variation_branches]
diagonal_flat = np.concatenate([values.ravel() for values in diagonal_variations])
print('Correlated covariance:\n', correlated_k6.covariance)
print('Diagonal extraction covariance:\n', diagonal_k6.covariance)
print('Marginal 1-sigma errors:', np.sqrt(np.diag(diagonal_k6.covariance)))
print('CV exactly matches correlated kmax=6:', np.array_equal(correlated_cv, diagonal_cv))
print('All shifted weights finite:', np.isfinite(diagonal_flat).all())
print('Negative shifted weights:', np.sum(diagonal_flat < 0))
print('Every zero shift equals CV:', all(np.array_equal(values[:, 3], diagonal_cv) for values in diagonal_variations))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
correlated_samples = rng.multivariate_normal(correlated_k6.free_a_values, correlated_k6.covariance, 4000)
diagonal_samples = rng.multivariate_normal(diagonal_k6.free_a_values, diagonal_k6.covariance, 4000)
axes[0].scatter(correlated_samples[:, 0], correlated_samples[:, 1], s=5, alpha=.12, label='reported covariance')
axes[0].scatter(diagonal_samples[:, 0], diagonal_samples[:, 1], s=5, alpha=.12, label='diagonal covariance')
axes[0].plot(*diagonal_k6.free_a_values, 'k*', ms=12, label='shared CV')
axes[0].set(xlabel=r'$a_1$', ylabel=r'$a_2$', title='Same center and widths; correlations removed')
axes[0].legend(fontsize=8)
for parameter_i, (branch, values) in enumerate(zip(diagonal_k6.variation_branches, diagonal_variations), start=1):
    axes[1].plot(ZEXP_SIGMA_VALUES, values.mean(axis=0), 'o-', lw=2, label=rf'direct $a_{parameter_i}$ shift')
axes[1].set(xlabel=r'Parameter shift ($\sigma$)', ylabel='Mean real-event weight', title=r'Direct $a_1$ and $a_2$ weight branches')
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## One construction path for every prior

In [ ]:
prior_summary = pd.DataFrame([
    {
        'prior': prior.name,
        'kmax': prior.kmax,
        'free parameters': len(prior.free_a_values),
        'variations': 'PCA' if prior.use_pca else 'direct a_k',
        'CV branch': prior.cv_branch,
    }
    for prior in ZEXP_PRIORS
])
display(prior_summary)

standardized_weights = {
    prior.name: compute_zexp_prior_weights(sample_q2, sample_ma, prior)
    for prior in ZEXP_PRIORS
}
assert all(
    set(weights) == {prior.cv_branch, *prior.variation_branches}
    for prior, weights in zip(ZEXP_PRIORS, standardized_weights.values())
)
print('All production priors pass through the same constructor and return their declared branches.')